In [44]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import seaborn as sn

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input/'):
    for filename in filenames:
        print(os.path.join(dirname,filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/digit-recognizer/sample_submission.csv
/kaggle/input/digit-recognizer/train.csv
/kaggle/input/digit-recognizer/test.csv


In [45]:
train_df = pd.read_csv('/kaggle/input/digit-recognizer/train.csv')
test_df = pd.read_csv('/kaggle/input/digit-recognizer/test.csv')    #this is the unseen data which we will impliment the model on
sample_df = pd.read_csv('/kaggle/input/digit-recognizer/sample_submission.csv')
train_df

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41995,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
41996,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
41997,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
41998,6,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [46]:
test_df

,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27995,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
27996,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
27997,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
27998,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [47]:
sample_df

,ImageId,Label
0,1,0
1,2,0
2,3,0
3,4,0
4,5,0
...,...,...
27995,27996,0
27996,27997,0
27997,27998,0
27998,27999,0


In [48]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
np.random.seed(1212)

from tensorflow import keras
from tensorflow.keras.models import Model
from tensorflow.keras.layers import *
from tensorflow.keras import optimizers

In [49]:
train_df.shape,test_df.shape

((42000, 785), (28000, 784))

In [50]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42000 entries, 0 to 41999
Columns: 785 entries, label to pixel783
dtypes: int64(785)
memory usage: 251.5 MB


In [51]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28000 entries, 0 to 27999
Columns: 784 entries, pixel0 to pixel783
dtypes: int64(784)
memory usage: 167.5 MB


In [52]:
train_df.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [53]:
train_df[['pixel9','pixel15']].head(20)

,pixel9,pixel15
0,0,0
1,0,0
2,0,0
3,0,0
4,0,0
5,0,0
6,0,0
7,0,0
8,0,0
9,0,0


In [54]:
train_df.loc[:10,'pixel9':'pixel15']

,pixel9,pixel10,pixel11,pixel12,pixel13,pixel14,pixel15
0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0
6,0,0,0,0,0,0,0
7,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0
9,0,0,0,0,0,0,0


In [55]:
train_df.describe()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
count,42000.000000,42000.0,42000.0,42000.0,42000.0,42000.0,42000.0,42000.0,42000.0,42000.0,...,42000.000000,42000.000000,42000.000000,42000.00000,42000.000000,42000.000000,42000.0,42000.0,42000.0,42000.0
mean,4.456643,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.219286,0.117095,0.059024,0.02019,0.017238,0.002857,0.0,0.0,0.0,0.0
std,2.887730,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,6.312890,4.633819,3.274488,1.75987,1.894498,0.414264,0.0,0.0,0.0,0.0
min,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.0
25%,2.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.0
50%,4.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.0
75%,7.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.0
max,9.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,254.000000,254.000000,253.000000,253.00000,254.000000,62.000000,0.0,0.0,0.0,0.0


In [56]:
test_df.describe()

,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
count,28000.0,28000.0,28000.0,28000.0,28000.0,28000.0,28000.0,28000.0,28000.0,28000.0,...,28000.000000,28000.000000,28000.000000,28000.000000,28000.000000,28000.0,28000.0,28000.0,28000.0,28000.0
mean,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.164607,0.073214,0.028036,0.011250,0.006536,0.0,0.0,0.0,0.0,0.0
std,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,5.473293,3.616811,1.813602,1.205211,0.807475,0.0,0.0,0.0,0.0,0.0
min,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
25%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
50%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
75%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
max,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,253.000000,254.000000,193.000000,187.000000,119.000000,0.0,0.0,0.0,0.0,0.0


# **split data into train and validation set** 

In [57]:
train_df

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41995,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
41996,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
41997,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
41998,6,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [58]:
X =  train_df.iloc[:,1:]
X

,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41995,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
41996,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
41997,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
41998,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [59]:
y_label = train_df.loc[:,'label']  # i am simply dropping the features and keeping the dependent variable, but i am using a different method that doesn't use the drop method
y_label 


0        1
1        0
2        1
3        4
4        0
        ..
41995    0
41996    1
41997    7
41998    6
41999    9
Name: label, Length: 42000, dtype: int64

In [60]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y_label,random_state = 1212, test_size = 0.2)

In [61]:
X_train.shape,X_test.shape,y_train.shape,y_test.shape

((33600, 784), (8400, 784), (33600,), (8400,))

In [62]:
X_train

,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
14176,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
16218,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
14355,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9215,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
41937,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17509,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
19010,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
23990,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
34458,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [63]:
# now i will bring in the unseen data for analysis
test_df.shape

(28000, 784)

In [64]:
X_train.shape,X_test.shape,test_df.shape

((33600, 784), (8400, 784), (28000, 784))

In [65]:
# reshape only the splited data for the features
X_train =  X_train.values.reshape(33600, 784)
X_test = X_test.values.reshape(8400, 784)
                              
# let's reshape the unseen test data as well

test_df = test_df.values.reshape(28000, 784)

# **Data Cleaning And Normalzation**

In [66]:
print((min(X_train[1]),max(X_train[1])))

(np.int64(0), np.int64(255))


### **pixel intensities are currently between the range of 0 and 255, let's proceed to normalize the features, using broadcasting operation(/=,+= etc)**

In [67]:
X_train = X_train.astype('float32');test_df = test_df.astype('float32'); X_test = X_test.astype('float32')
X_train/=255;X_test/=255;test_df/=255


 **convert labels from a class vector to binary One Hot Encoded**
 Neural networks don’t work well with single integer labels for classification.

The network expects one-hot encoded vectors.

One-hot encoding converts a class like 3 into a vector of length 10:

3 → [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
Now each label is a vector, ready for neural network output.

In [68]:
digit_numbers = 10
y_train = keras.utils.to_categorical(y_train,digit_numbers)
y_test = keras.utils.to_categorical(y_test,digit_numbers)





*Both need to be one-hot encoded because your output layer of the neural network has:Dense(10, activation='softmax')Output = 10 probabilities (one for each digit)Loss function = categorical_crossentropyGround truth = one-hot vectorWithout one-hot encoding, the network cannot compute the loss properly*


In [69]:
# Printing 2 examples of labels after conversion
print(y_train[0]) # 2
print(y_train[3]) # 7

[0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]


i wll proceed by fitting several simple neural network models using Keras (with TensorFlow as our backend) and collect their accuracy. The model that performs the best on the validation set will be used as the model of choice for the competition.

Model 1: Simple Neural Network with 4 layers (300, 100, 100, 200)

In our first model, we will use the Keras library to train a neural network with the activation function set as ReLu. To determine which class to output, we will rely on the SoftMax function.  softmax  is best for multiple outputting. there is sigmoid function as etc.

In [70]:
# Input Parameters
n_input = 784 # number of features
n_hidden_1 = 300
n_hidden_2 = 100
n_hidden_3 = 100
n_hidden_4 = 200
num_digits = 10
verbose =  2


In [71]:
Inp = Input(shape=(784,))
x = Dense(n_hidden_1, activation='relu', name = "Hidden_Layer_1")(Inp)
x = Dense(n_hidden_2, activation='relu', name = "Hidden_Layer_2")(x)
x = Dense(n_hidden_3, activation='relu', name = "Hidden_Layer_3")(x)
x = Dense(n_hidden_4, activation='relu', name = "Hidden_Layer_4")(x)
output = Dense(num_digits, activation='softmax', name = "Output_Layer")(x)

In [72]:
model= Model(Inp,output )
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Hidden_Layer_1 (Dense)          │ (None, 300)            │       235,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Hidden_Layer_2 (Dense)          │ (None, 100)            │        30,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Hidden_Layer_3 (Dense)          │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Hidden_Layer_4 (Dense)          │ (None, 200)            │        20,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output_Layer (Dense)            │ (None, 10)             │         2,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 297,910 (1.14 MB)

 Trainable params: 297,910 (1.14 MB)

 Non-trainable params: 0 (0.00 B)

In [73]:
# insert hyperparameters
# these are actually parameters we will be changing often to check model performance

learning_rate = 0.1
training_epochs = 20
verbose = 2
batch_size =100  #The model will process 100 training samples at a time before updating the weights.
sgd = optimizers.SGD(learning_rate = learning_rate)
adam = keras.optimizers.Adam(learning_rate=learning_rate)
verbose = 2

                

In [74]:
# plain vanilla Stochastic Gradient Descent as our optimizing methodology
model.compile(loss='categorical_crossentropy',
              optimizer='sgd',
              metrics=['accuracy'])

In [75]:
history1 = model.fit(X_train, y_train,
                     batch_size = batch_size,
                     epochs = training_epochs,
                     verbose = 2,
                     validation_data=(X_test, y_test))

Epoch 1/20
336/336 - 3s - 8ms/step - accuracy: 0.4937 - loss: 1.7974 - val_accuracy: 0.7727 - val_loss: 0.9161
Epoch 2/20
336/336 - 2s - 5ms/step - accuracy: 0.8414 - loss: 0.5945 - val_accuracy: 0.8755 - val_loss: 0.4391
Epoch 3/20
336/336 - 2s - 5ms/step - accuracy: 0.8863 - loss: 0.3939 - val_accuracy: 0.8970 - val_loss: 0.3534
Epoch 4/20
336/336 - 2s - 5ms/step - accuracy: 0.9051 - loss: 0.3288 - val_accuracy: 0.9092 - val_loss: 0.3110
Epoch 5/20
336/336 - 2s - 5ms/step - accuracy: 0.9159 - loss: 0.2914 - val_accuracy: 0.9196 - val_loss: 0.2805
Epoch 6/20
336/336 - 2s - 5ms/step - accuracy: 0.9238 - loss: 0.2648 - val_accuracy: 0.9265 - val_loss: 0.2562
Epoch 7/20
336/336 - 2s - 5ms/step - accuracy: 0.9299 - loss: 0.2427 - val_accuracy: 0.9279 - val_loss: 0.2521
Epoch 8/20
336/336 - 2s - 5ms/step - accuracy: 0.9353 - loss: 0.2246 - val_accuracy: 0.9336 - val_loss: 0.2308
Epoch 9/20
336/336 - 2s - 5ms/step - accuracy: 0.9391 - loss: 0.2091 - val_accuracy: 0.9369 - val_loss: 0.2154
E

In [76]:
predictions = model.predict(X_test)

263/263 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [77]:
# Convert probabilities to digit labels
predictions.shape

(8400, 10)

now let's improve our predictions by changing just the optimizers. in this project we won't change all the hyperparameters but we will focus on improving the model by only changing or adjusting the activation 

In [78]:
model2a = Model(Inp, output)
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Hidden_Layer_1 (Dense)          │ (None, 300)            │       235,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Hidden_Layer_2 (Dense)          │ (None, 100)            │        30,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Hidden_Layer_3 (Dense)          │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Hidden_Layer_4 (Dense)          │ (None, 200)            │        20,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output_Layer (Dense)            │ (None, 10)             │         2,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 297,912 (1.14 MB)

 Trainable params: 297,910 (1.14 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [79]:
model2a.compile(loss='categorical_crossentropy',
               optimizer = 'adam',
               metrics =['accuracy'])

In [80]:
history2 = model2a.fit(X_train,y_train,
                       batch_size = batch_size,
                       epochs = training_epochs,
                       verbose = verbose,
                       validation_data = (X_test, y_test))


Epoch 1/20
336/336 - 4s - 11ms/step - accuracy: 0.9513 - loss: 0.1556 - val_accuracy: 0.9542 - val_loss: 0.1458
Epoch 2/20
336/336 - 2s - 6ms/step - accuracy: 0.9715 - loss: 0.0914 - val_accuracy: 0.9504 - val_loss: 0.1650
Epoch 3/20
336/336 - 2s - 6ms/step - accuracy: 0.9784 - loss: 0.0695 - val_accuracy: 0.9673 - val_loss: 0.1083
Epoch 4/20
336/336 - 2s - 6ms/step - accuracy: 0.9846 - loss: 0.0484 - val_accuracy: 0.9720 - val_loss: 0.1030
Epoch 5/20
336/336 - 2s - 6ms/step - accuracy: 0.9857 - loss: 0.0421 - val_accuracy: 0.9708 - val_loss: 0.1037
Epoch 6/20
336/336 - 2s - 6ms/step - accuracy: 0.9890 - loss: 0.0333 - val_accuracy: 0.9729 - val_loss: 0.0989
Epoch 7/20
336/336 - 2s - 6ms/step - accuracy: 0.9905 - loss: 0.0281 - val_accuracy: 0.9675 - val_loss: 0.1397
Epoch 8/20
336/336 - 2s - 6ms/step - accuracy: 0.9915 - loss: 0.0244 - val_accuracy: 0.9725 - val_loss: 0.1199
Epoch 9/20
336/336 - 2s - 6ms/step - accuracy: 0.9916 - loss: 0.0255 - val_accuracy: 0.9742 - val_loss: 0.1138


# lets adjust changing rate for model2a

As it turns out, it does appear to be the case that the optimizer plays a crucial part in the validation score. In particular, the model which relies on 'Adam' as its optimizer tend to perform 1.5 - 2.5% better on average. Going forward, we will use 'Adam' as our optimizer of choice.

What if we changed the learning rate from 0.1 to 0.01, or 0.5? Will it have any impact on the accuracy? Model 2A

In [81]:
new_learning_rate = 0.01
adam_new = keras.optimizers.Adam(learning_rate = new_learning_rate)

In [82]:
model2a_n = Model(Inp, output)
model2a_n.compile(loss = 'categorical_crossentropy',
                 optimizer = adam_new,
                 metrics = ['accuracy'])

In [83]:
history2a = model2a_n.fit(X_train, y_train,
                        batch_size = batch_size,
                        epochs = training_epochs,
                        verbose = verbose,
                        validation_data=(X_test, y_test))

Epoch 1/20
336/336 - 4s - 11ms/step - accuracy: 0.9474 - loss: 0.1994 - val_accuracy: 0.9598 - val_loss: 0.1637
Epoch 2/20
336/336 - 2s - 6ms/step - accuracy: 0.9612 - loss: 0.1445 - val_accuracy: 0.9621 - val_loss: 0.1594
Epoch 3/20
336/336 - 2s - 6ms/step - accuracy: 0.9618 - loss: 0.1493 - val_accuracy: 0.9613 - val_loss: 0.1788
Epoch 4/20
336/336 - 2s - 6ms/step - accuracy: 0.9699 - loss: 0.1210 - val_accuracy: 0.9629 - val_loss: 0.1472
Epoch 5/20
336/336 - 2s - 6ms/step - accuracy: 0.9724 - loss: 0.1124 - val_accuracy: 0.9681 - val_loss: 0.1520
Epoch 6/20
336/336 - 2s - 6ms/step - accuracy: 0.9762 - loss: 0.0928 - val_accuracy: 0.9724 - val_loss: 0.1282
Epoch 7/20
336/336 - 2s - 6ms/step - accuracy: 0.9797 - loss: 0.0770 - val_accuracy: 0.9707 - val_loss: 0.1306
Epoch 8/20
336/336 - 2s - 6ms/step - accuracy: 0.9826 - loss: 0.0648 - val_accuracy: 0.9677 - val_loss: 0.1539
Epoch 9/20
336/336 - 2s - 6ms/step - accuracy: 0.9778 - loss: 0.0891 - val_accuracy: 0.9739 - val_loss: 0.1312


In [84]:
pred_for_m2a_n = model2a.predict(X_test)

263/263 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [85]:
pred_for_m2a_n.shape

(8400, 10)

In [86]:
import numpy as np

pred_for_m2a_n = model2a_n.predict(X_test)
pred_for_m2a_n = np.argmax(pred_for_m2a_n, axis=1)

submission = pd.DataFrame({
    "ImageId": range(1, len(pred_for_m2a_n) + 1),
    "Label": pred_for_m2a_n
})

submission.to_csv("submission_newIDN.csv", index=False)
print("Submission file created!")


263/263 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Submission file created!


**note, adding an additional layer does not significantly improve the accuracy of a model. However, there are computational costs (in terms of complexity) in implementing an additional layer in neural network. Given that the benefits of an additional layer are low while the costs are high,always  stick with the 4 layers neural network or less.**
 **Also, the accuracy, as measured by the 3 different learning rates 0.01, 0.1 and 0.5 are around 98%, 97% and 98% respectively. As there are no considerable gains by changing the learning rates, it is best to stick with the default learning rate of 0.01.**

** now  i will proceed to include dropout (dropout rate of 0.3) in our second model to prevent overfitting.**
note also that dropout is added to the early layers, this helps to reduce noice and prevent overfitting. in some cases adding it to the last layers can optimize the accuracyb ut best practice is to add it in early layers

Now let's control overfitting by  introducing dropout layer

In [87]:
# input parameters
n_input =  784
hidden_1 = 300
hidden_2 = 100
hidden_3 = 100
hidden_4 = 200
num_digits = 10

In [88]:
INP = Input(shape=(784,))
x = Dense(hidden_1,activation = 'relu',name = 'hidden_layer_1')(INP)
x=Dropout(0.3)(x)
x = Dense(hidden_2,activation = 'relu',name = 'hidden_layer_2')(x)
x = Dropout(0.3)(x)
x = Dense(hidden_3,activation = 'relu',name = 'hidden_layer_3')(INP)
x=Dropout(0.3)(x)
x = Dense(hidden_4,activation = 'relu',name = 'hidden_layer_4')(INP)
output = Dense(num_digits,activation ='softmax', name = 'output_layer')(x)
                


In [89]:
model3 = Model(INP, output)
model3.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ hidden_layer_4 (Dense)          │ (None, 200)            │       157,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 10)             │         2,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 159,010 (621.13 KB)

 Trainable params: 159,010 (621.13 KB)

 Non-trainable params: 0 (0.00 B)

In [90]:
# i will compile the model with adam optimizer
model3.compile(loss='categorical_crossentropy',
              optimizer = keras.optimizers.Adam(learning_rate = 0.01),
              metrics = ['accuracy'])

In [91]:
model3.fit(X_train, y_train,
          batch_size = batch_size,
          epochs = training_epochs,
          validation_data = (X_test, y_test))

Epoch 1/20
336/336 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.8655 - loss: 0.4188 - val_accuracy: 0.9560 - val_loss: 0.1519
Epoch 2/20
336/336 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9621 - loss: 0.1225 - val_accuracy: 0.9630 - val_loss: 0.1379
Epoch 3/20
336/336 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9723 - loss: 0.0869 - val_accuracy: 0.9651 - val_loss: 0.1382
Epoch 4/20
336/336 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9777 - loss: 0.0727 - val_accuracy: 0.9646 - val_loss: 0.1320
Epoch 5/20
336/336 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9805 - loss: 0.0623 - val_accuracy: 0.9685 - val_loss: 0.1422
Epoch 6/20
336/336 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9848 - loss: 0.0486 - val_accuracy: 0.9602 - val_loss: 0.1918
Epoch 7/20
336/336 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9842 - loss: 0.0515 - val_accuracy: 0.9671 - val_loss: 0.1796
Epoch 8/20
336/336 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9826 - loss: 0.0563 - val_accuracy: 0.

In [92]:
predictions_main = model3.predict(test_df, batch_size = 200)

140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [93]:
pred_for_m3 = np.argmax(predictions_main, axis=1)

submission = pd.DataFrame({
    "ImageId": range(1, len(pred_for_m3) + 1),
    "Label": pred_for_m3
})

submission.to_csv("/kaggle/working/submission_IDNE.csv", index=False)
print("Submission file created!")



Submission file created!
